1 - importação das bibliotecas

In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
import requests
pasta_books = "books" # variavel de localização dos livros

1 - coleta dos IDs de cada livro com base no ID disponivel no site: https://www.gutenberg.org/

- ao analisar os livros, percebi que todos vem da mesma fonte, do projeto gutenberg.
- devido virem da mesma fonte, decidi que usarei o "ID" dos livros anexados para o codigo
- para coletar o ID, usairei o valor disponivel em cada link dos livros
- nessa linha tambem criei o dataframe df_livros

In [2]:
id_coletado = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            meta_downloads = soup.find('link', {'rel': 'dcterms.isFormatOf'})
            id_livro = None
            
            if meta_downloads and meta_downloads.get('href'):
                url = meta_downloads['href'].strip() # aqui coleto toda a url disponivel no livro
                match = re.search(r'/ebooks/(\d+)', url) # aqui eu coleto apenas o numero do livro que usarei para o ID
                if match:
                    id_livro = int(match.group(1))
            
            id_coletado.append(id_livro)

df_livros = pd.DataFrame({"ID": id_coletado})

df_livros["ID"] = df_livros["ID"].astype("Int64")

df_livros.head(15)

,ID
0,2641
1,76
2,11
3,84
4,1342
5,1513
6,1661
7,67979
8,16389
9,64317


2 - coleta do titulo dos livros:

- todos os livros apresentam uma TAG META com o nome "dc.title" contendo o titulo de cada livro, com isso em mente, uso o codigo abaixo para realizar um scrap em cada livro.

In [3]:
titulo_coletado = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            meta_titulo = soup.find('meta', {'name': 'dc.title'})
            if meta_titulo and meta_titulo.get('content'):
                titulo = meta_titulo['content'].strip()
            else:
                titulo = "Sem Título"
            
            titulo_coletado.append(titulo)

df_livros["Titulo"] = titulo_coletado

df_livros["Titulo"] = df_livros["Titulo"].astype("string")

df_livros.head(15)

,ID,Titulo
0,2641,A Room with a View
1,76,Adventures of Huckleberry Finn
2,11,Alice's Adventures in Wonderland
3,84,"Frankenstein; or, the modern prometheus"
4,1342,Pride and Prejudice
5,1513,Romeo and Juliet
6,1661,The Adventures of Sherlock Holmes
7,67979,The Blue Castle: a novel
8,16389,The Enchanted April
9,64317,The Great Gatsby


3 - coleta dos autores dos livros

- utilizando a mesma ideia, todos os livros apresentam a TAG META com nome "dc.creator".
- nessa mesma TAG, temos o ano de nascimento e de falecimento de cada autor. ambos dados serão armazenados no dataframe, porem, serão separados em colunas
- para separar o tempo de vida e o nome do autor, eu usei a biblioteca "re".  ao usar o re.search(), eu consigo separar o ano de vida do nome do autor

In [4]:
autores_coletados = []
periodos_coletados = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')

            meta_author = soup.find('meta', {'name': 'dc.creator'})
            
            if meta_author and meta_author.get('content'):
                autor_bruto = meta_author['content'].strip() # variavel contendo o nome do autor e o ano de vida ex: Doyle, Arthur Conan, 1859-1930
                
                coleta_periodo = re.search(r'\d{4}-\d{4}|\d{4}-', autor_bruto) # a variavel recebe apenas o ano de nascimento e falecimento
                
                if coleta_periodo:

                    periodo = coleta_periodo.group().strip() 

                    autor_sem_ano = autor_bruto.replace(periodo, "") # nessa linha, a parte do periodo e substituida por uma string vazia

                    autor_limpo = re.sub(r',\s*$', '', autor_sem_ano).strip() # apenas para remover simbolos desnecessarios do nome
                else:
                    periodo = "Não Informado"
                    autor_limpo = autor_bruto
            else:
                autor_limpo = "Autor Desconhecido"
                periodo = "Não Informado"
            
            autores_coletados.append(autor_limpo)
            periodos_coletados.append(periodo)

df_livros["Autor"] = autores_coletados
df_livros["Periodo"] = periodos_coletados

df_livros["Autor"] = df_livros["Autor"].astype("string")
df_livros["Periodo"] = df_livros["Periodo"].astype("string")

df_livros.head(15)

,ID,Titulo,Autor,Periodo
0,2641,A Room with a View,"Forster, E. M. (Edward Morgan)",1879-1970
1,76,Adventures of Huckleberry Finn,"Twain, Mark",1835-1910
2,11,Alice's Adventures in Wonderland,"Carroll, Lewis",1832-1898
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft",1797-1851
4,1342,Pride and Prejudice,"Austen, Jane",1775-1817
5,1513,Romeo and Juliet,"Shakespeare, William",1564-1616
6,1661,The Adventures of Sherlock Holmes,"Doyle, Arthur Conan",1859-1930
7,67979,The Blue Castle: a novel,"Montgomery, L. M. (Lucy Maud)",1874-1942
8,16389,The Enchanted April,"Von Arnim, Elizabeth",1866-1941
9,64317,The Great Gatsby,"Fitzgerald, F. Scott (Francis Scott)",1896-1940


4 - coleta do idioma de cada livro

- nessa etapa, coleto a TAG de nome "dc.language".

In [5]:
idiomas_coletados = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            meta_lang = soup.find('meta', {'name': 'dc.language'})
            
            if meta_lang and meta_lang.get('content'):
                idioma = meta_lang['content'].strip() 
            else:
                idioma = "não identificado"
            
            idiomas_coletados.append(idioma)

df_livros["Idioma"] = idiomas_coletados

df_livros["Idioma"] = df_livros["Idioma"].astype("string")

df_livros.head(15)

,ID,Titulo,Autor,Periodo,Idioma
0,2641,A Room with a View,"Forster, E. M. (Edward Morgan)",1879-1970,en
1,76,Adventures of Huckleberry Finn,"Twain, Mark",1835-1910,en
2,11,Alice's Adventures in Wonderland,"Carroll, Lewis",1832-1898,en
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft",1797-1851,en
4,1342,Pride and Prejudice,"Austen, Jane",1775-1817,en
5,1513,Romeo and Juliet,"Shakespeare, William",1564-1616,en
6,1661,The Adventures of Sherlock Holmes,"Doyle, Arthur Conan",1859-1930,en
7,67979,The Blue Castle: a novel,"Montgomery, L. M. (Lucy Maud)",1874-1942,en
8,16389,The Enchanted April,"Von Arnim, Elizabeth",1866-1941,en
9,64317,The Great Gatsby,"Fitzgerald, F. Scott (Francis Scott)",1896-1940,en


5 - coleta do genero de cada livro

- para coleta do genero, utilizei a TAG "dc.subject".
- ao analisar os livros, percebi que eles apresentam multiplos generos, porem, o primeiro parece ser mais importante, por isso, eu realizo a coleta apenas do primeiro genero.

In [6]:
generos_coletados = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        
        caminho_completo = os.path.join(pasta_books, arquivo)

        with open(caminho_completo, 'r', encoding='utf-8') as f:
            
            soup = BeautifulSoup(f.read(), 'html.parser')

            meta_subjects = soup.find_all('meta', {'name': 'dc.subject'})

            if meta_subjects and meta_subjects[0].get('content'):  # como há varios generos nos livros, coloquei para que pegasse apenas o primeiro genero
                genero = meta_subjects[0]['content'].strip()
            else:
                genero = "genero não identificado"

            generos_coletados.append(genero)

df_livros["Genero"] = generos_coletados

df_livros["Genero"] = df_livros["Genero"].astype("string")

df_livros.head(15)

,ID,Titulo,Autor,Periodo,Idioma,Genero
0,2641,A Room with a View,"Forster, E. M. (Edward Morgan)",1879-1970,en,Humorous stories
1,76,Adventures of Huckleberry Finn,"Twain, Mark",1835-1910,en,Humorous stories
2,11,Alice's Adventures in Wonderland,"Carroll, Lewis",1832-1898,en,Fantasy fiction
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft",1797-1851,en,Science fiction
4,1342,Pride and Prejudice,"Austen, Jane",1775-1817,en,England -- Fiction
5,1513,Romeo and Juliet,"Shakespeare, William",1564-1616,en,Vendetta -- Drama
6,1661,The Adventures of Sherlock Holmes,"Doyle, Arthur Conan",1859-1930,en,"Holmes, Sherlock (Fictitious character) -- Fic..."
7,67979,The Blue Castle: a novel,"Montgomery, L. M. (Lucy Maud)",1874-1942,en,Self-actualization (Psychology) -- Fiction
8,16389,The Enchanted April,"Von Arnim, Elizabeth",1866-1941,en,Love stories
9,64317,The Great Gatsby,"Fitzgerald, F. Scott (Francis Scott)",1896-1940,en,Psychological fiction


6 - coleta da publicação

- ao analisar os livros, não encontrei uma data especifica que os livros foram publicados, porem, há uma TAG informando uma data em especifico, então decidi usa-la.
- nesse caso, usei a TAG "dcterms.created". nessa tag, há o dia e o mes, porem, irei colocar apenas o ano de publicação.

In [7]:
anos_publicacao = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            meta_created = soup.find('meta', {'name': 'dcterms.created'})
            
            if meta_created and meta_created.get('content'):
                data_bruta = meta_created['content'].strip()
                
                match_ano = re.search(r'\d{4}', data_bruta)  # codigo para remover as datas e colocar apenas o ano de lançamento.
                ano = match_ano.group()
            else:
                ano = "Não Informado"
                
            anos_publicacao.append(ano)

df_livros["Ano_Publicacao"] =  pd.Series(anos_publicacao, dtype="Int64")

df_livros.head(15)

,ID,Titulo,Autor,Periodo,Idioma,Genero,Ano_Publicacao
0,2641,A Room with a View,"Forster, E. M. (Edward Morgan)",1879-1970,en,Humorous stories,2001
1,76,Adventures of Huckleberry Finn,"Twain, Mark",1835-1910,en,Humorous stories,2004
2,11,Alice's Adventures in Wonderland,"Carroll, Lewis",1832-1898,en,Fantasy fiction,2008
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft",1797-1851,en,Science fiction,1993
4,1342,Pride and Prejudice,"Austen, Jane",1775-1817,en,England -- Fiction,1998
5,1513,Romeo and Juliet,"Shakespeare, William",1564-1616,en,Vendetta -- Drama,1998
6,1661,The Adventures of Sherlock Holmes,"Doyle, Arthur Conan",1859-1930,en,"Holmes, Sherlock (Fictitious character) -- Fic...",1999
7,67979,The Blue Castle: a novel,"Montgomery, L. M. (Lucy Maud)",1874-1942,en,Self-actualization (Psychology) -- Fiction,2022
8,16389,The Enchanted April,"Von Arnim, Elizabeth",1866-1941,en,Love stories,2005
9,64317,The Great Gatsby,"Fitzgerald, F. Scott (Francis Scott)",1896-1940,en,Psychological fiction,2021


7 - coleta do numero de capitulos

- essa parte foi a mais complexa do codigo. devido quase todos os livros terem formas diferentes de indicar os capitulos. mas no geral, a grande parte dos livros apresentam uma TAG "a" com uma referencia "href" indicando o capitulo e sua enumeração.
- apartir dessa analise, fiz o codigo para contar cada href com as iniciais: #link2HCH, #chap, #CHAPTER_, #scene, #chapter-.
- para evitar duplicatas criei o conjunto links_vistos()  para armazenar itens unicos, evitando repetições.
- infelizmente, há livros que não contem o padrão de capitulos em links. nesse caso, esses livros recebem o valor nulo.

In [8]:
total_capitulos_por_livro = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            links_vistos = set()  # criei esse conjunto para listar apenas itens unicos
            todas_tags_a = soup.find_all('a', href=True) # procura por tags "a" que contenham href
            
            for tag in todas_tags_a:
                href = tag['href'].strip()
                
                if href.startswith(('#link2HCH', '#chap', '#CHAPTER_', '#scene', '#chapter-')): # condição para contar apenas as referencias descritas
                    links_vistos.add(href) # o conjunto recebe os capitulos sem recontar com capitulos repetidos

            if links_vistos:
                total_capitulos = len(links_vistos) # aqui eu contabilizo o numero de capitulos diferenciais
            else:
                total_capitulos = None # caso tenha 0 capitulos, o livro recebe um valor nulo
                
            total_capitulos_por_livro.append(total_capitulos)

df_livros["Total_Capitulos"] = pd.Series(total_capitulos_por_livro, dtype="Int64") # salvei os dados de capitulos no dataframe e para ter valores nulos, transformei a coluna em inteiros

df_livros.head(15)

,ID,Titulo,Autor,Periodo,Idioma,Genero,Ano_Publicacao,Total_Capitulos
0,2641,A Room with a View,"Forster, E. M. (Edward Morgan)",1879-1970,en,Humorous stories,2001,20
1,76,Adventures of Huckleberry Finn,"Twain, Mark",1835-1910,en,Humorous stories,2004,43
2,11,Alice's Adventures in Wonderland,"Carroll, Lewis",1832-1898,en,Fantasy fiction,2008,12
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft",1797-1851,en,Science fiction,1993,24
4,1342,Pride and Prejudice,"Austen, Jane",1775-1817,en,England -- Fiction,1998,60
5,1513,Romeo and Juliet,"Shakespeare, William",1564-1616,en,Vendetta -- Drama,1998,26
6,1661,The Adventures of Sherlock Holmes,"Doyle, Arthur Conan",1859-1930,en,"Holmes, Sherlock (Fictitious character) -- Fic...",1999,12
7,67979,The Blue Castle: a novel,"Montgomery, L. M. (Lucy Maud)",1874-1942,en,Self-actualization (Psychology) -- Fiction,2022,45
8,16389,The Enchanted April,"Von Arnim, Elizabeth",1866-1941,en,Love stories,2005,22
9,64317,The Great Gatsby,"Fitzgerald, F. Scott (Francis Scott)",1896-1940,en,Psychological fiction,2021,9


8 - coleta dos direitos autorais

- seguindo as ideias anteriores, nessa etapa eu coleto a TAG meta com nome "dc.rights" para coletar os direitos sobre o livro

In [9]:
direitos_coletados = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            meta_rights = soup.find('meta', {'name': 'dc.rights'})
            
            if meta_rights and meta_rights.get('content'):
                direitos = meta_rights['content'].strip()
            else:
                direitos = "copyrights não identificado"
            
            direitos_coletados.append(direitos)

df_livros["Direitos"] = direitos_coletados

df_livros["Direitos"] = df_livros["Direitos"].astype("string")

df_livros.head(15)

,ID,Titulo,Autor,Periodo,Idioma,Genero,Ano_Publicacao,Total_Capitulos,Direitos
0,2641,A Room with a View,"Forster, E. M. (Edward Morgan)",1879-1970,en,Humorous stories,2001,20,Public domain in the USA.
1,76,Adventures of Huckleberry Finn,"Twain, Mark",1835-1910,en,Humorous stories,2004,43,Public domain in the USA.
2,11,Alice's Adventures in Wonderland,"Carroll, Lewis",1832-1898,en,Fantasy fiction,2008,12,Public domain in the USA.
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft",1797-1851,en,Science fiction,1993,24,Public domain in the USA.
4,1342,Pride and Prejudice,"Austen, Jane",1775-1817,en,England -- Fiction,1998,60,Public domain in the USA.
5,1513,Romeo and Juliet,"Shakespeare, William",1564-1616,en,Vendetta -- Drama,1998,26,Public domain in the USA.
6,1661,The Adventures of Sherlock Holmes,"Doyle, Arthur Conan",1859-1930,en,"Holmes, Sherlock (Fictitious character) -- Fic...",1999,12,Public domain in the USA.
7,67979,The Blue Castle: a novel,"Montgomery, L. M. (Lucy Maud)",1874-1942,en,Self-actualization (Psychology) -- Fiction,2022,45,Public domain in the USA.
8,16389,The Enchanted April,"Von Arnim, Elizabeth",1866-1941,en,Love stories,2005,22,Public domain in the USA.
9,64317,The Great Gatsby,"Fitzgerald, F. Scott (Francis Scott)",1896-1940,en,Psychological fiction,2021,9,Public domain in the USA.


9 - coleta dos downloads dos livros

- nessa etapa, utilizei a tag link para coleta dos downloads, percebi que varios livros apresentavam essa tag com o link de download.

In [ ]:
links_coletados = []

for arquivo in os.listdir(pasta_books):
    if arquivo.endswith(".html"):
        caminho_completo = os.path.join(pasta_books, arquivo)
        
        with open(caminho_completo, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            
            meta_downloads = soup.find('link', {'rel': 'dcterms.isFormatOf'})
            
            if meta_downloads and meta_downloads.get('href'):
                downloads = meta_downloads['href'].strip()
            else:
                downloads = "link não encontrado"

            links_coletados.append(downloads)

df_livros["Link_Download"] = links_coletados

df_livros["Link_Download"] = df_livros["Link_Download"].astype("string")

df_livros.head(15)

10 - salvamento do dataframe

In [11]:
arquivo = "data/desafio_HTML.csv"

df_livros.to_csv(arquivo, index=False, encoding='utf-8-sig')

In [12]:
df_livros.dtypes

ID                          Int64
Titulo             string[python]
Autor              string[python]
Periodo            string[python]
Idioma             string[python]
Genero             string[python]
Ano_Publicacao              Int64
Total_Capitulos             Int64
Direitos           string[python]
Link_Download      string[python]
dtype: object